# E08 · Optical flow and occupancy grids

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E08_optical_flow_and_occupancy_grids.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 60 min

**Skill Hub concepts:** Optical flow; Occupancy grid

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E08")

## Map: motion from pixels, maps from rays

**Brightness constancy:** a point keeps its intensity as it moves, $I(x + u, y + v, t + 1) = I(x, y, t)$. First-order
Taylor expansion gives one equation per pixel with two unknowns:
$$I_x u + I_y v + I_t = 0$$
**Lucas-Kanade** assumes the flow is constant in a window and solves the least-squares system with the
**structure tensor** $G$:
$$\underbrace{\begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix}}_{G}
\begin{bmatrix} u \\ v \end{bmatrix} = -\begin{bmatrix} \sum I_x I_t \\ \sum I_y I_t \end{bmatrix}$$

| Window content | Eigenvalues of $G$ | Rank | Flow recoverable |
|---|---|---|---|
| Flat region | both near 0 | 0 | nothing |
| Straight edge | one large, one near 0 | 1 | only the component normal to the edge (**aperture problem**) |
| Corner or blob | both large | 2 | both components (good feature to track: Shi-Tomasi score $=\lambda_{min}$) |

Large motions break the linearization: pyramids (coarse to fine) and iterations fix that.

**Occupancy grid:** each cell stores the **log-odds** $\ell = \ln\frac{p}{1-p}$ of being occupied. A range beam
updates every cell it crosses (free, $\ell \mathrel{+}= \ell_{free} < 0$) and the cell it ends in
(occupied, $\ell \mathrel{+}= \ell_{occ} > 0$). Additions replace Bayes products; clamping to $\pm\ell_{max}$ keeps
the map able to change when the world changes. Back to probability: $p = 1 / (1 + e^{-\ell})$.

In [ ]:
import numpy as np

## Task 1: structure tensor and Lucas-Kanade
Images are indexed `I[y, x]`. Use `np.gradient` on the first image for $I_x$ (axis 1) and $I_y$ (axis 0), and
$I_t = I_1 - I_0$. One window = the whole image.

In [ ]:
def structure_tensor(I):
    raise NotImplementedError  # TODO

def lucas_kanade(I0, I1):
    """Return (u, v): motion in pixels along x (columns) and y (rows)."""
    raise NotImplementedError  # TODO

lab.check("flow", structure_tensor, lucas_kanade)

## Task 2: ray casting into an occupancy grid
`bresenham(x0, y0, x1, y1)` returns the list of integer cells from start to end inclusive (any octant).
`update_ray(L, start, end, l_occ, l_free, l_max)` adds `l_free` to every cell before `end`, `l_occ` to `end`,
clamps to $\pm$`l_max`, and returns `L` (indexed `L[ix, iy]`).

In [ ]:
def bresenham(x0, y0, x1, y1):
    raise NotImplementedError  # TODO

def update_ray(L, start, end, l_occ=0.85, l_free=-0.4, l_max=3.5):
    raise NotImplementedError  # TODO

lab.check("occupancy", bresenham, update_ray)

In [ ]:
try:
    import matplotlib.pyplot as plt
    L = np.zeros((40, 40))
    for k in range(3):
        for a in np.linspace(0, 2 * np.pi, 180, endpoint=False):
            r = 0.0
            while True:
                r += 0.25
                x, y = int(round(20 + r * np.cos(a))), int(round(20 + r * np.sin(a)))
                if x in (5, 34) or y in (5, 34):
                    break
            update_ray(L, (20, 20), (x, y))
    plt.imshow(1 / (1 + np.exp(-L.T)), origin="lower", cmap="gray_r"); plt.title("p(occupied) after 3 sweeps"); plt.colorbar(); plt.show()
except NotImplementedError:
    print("finish Task 2 first")

## Checkpoint: Math
**M1 (Apply).** An inverse sensor model says a hit cell is occupied with probability 0.7. Log-odds increment $\ell_{occ}$?

**M2 (Apply).** A cell has log-odds 2.0. Probability it is occupied?

**M3 (Analyze).** Rank of the structure tensor of a window containing only one straight edge?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** A camera watches a long horizontal edge (a table edge) that moves diagonally. Lucas-Kanade on a
window containing only that edge recovers:
- a) both components of the motion
- b) only the vertical component, normal to the edge
- c) only the horizontal component, along the edge

**K2 (Evaluate).** Why store log-odds instead of probabilities, and why clamp at $\pm\ell_{max}$? Give the failure you
would see on a robot without the clamp.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e08.h`).
3. In `roboprep/checks/e08.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()